In [1]:
import numpy as np
import pandas as pd

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM,
    Dense
)

from tensorflow.keras.callbacks import (
    EarlyStopping
)

from tensorflow.keras.optimizers import Adam

In [2]:
%run 00.ipynb

c:\Users\SafinSharma\miniconda3\envs\forecasting\Lib\site-packages\nbformat\__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Export Folder : standardized_exports
Results Folder: results
24
Datasets Found: 22
agri_medium
agri_small
energy_medium
energy_small
favorita-train_medium
favorita-train_medium_cov
favorita-train_small
favorita-train_small_cov
favorita-transaction_medium
favorita-transaction_small
m4-daily_medium
m4-daily_small
m4-hourly_medium
m4-hourly_small
m4-quarterly_medium
m4-quarterly_small
m4-weekly_medium
m4-weekly_small
m4-yearly_medium
m4-yearly_small
traffic_medium
traffic_small
5
agri_medium               rows=15343      series=246      horizon=5
5
agri_small                rows=14428      series=246      horizon=5
24
energy_medium             rows=177600     series=370      horizon=24
24
energy_small              rows=44400      series=370      horizon=24
28
favorita-train_medium     rows=1985155    series=4504     horizon=28
favorita-train_medium_cov -> ERROR: No horizon configured for favorita-train_medium_cov
28
favorita-train_small      rows=609961     series=4504     horizon=28
favo

In [3]:
RESULT_FILE = "dl_results.csv"

initialize_results_file(
    RESULT_FILE
)

completed_runs = get_completed_runs(
    RESULT_FILE
)

print(
    f"Completed Runs: {len(completed_runs)}"
)

Completed Runs: 5


In [4]:
LOOKBACK = 14

def create_sequences(
    train_df,
    lookback=30
):

    X = []
    y = []

    for _, group in train_df.groupby(
        "unique_id"
    ):

        values = (
            group["y"]
            .astype(float)
            .values
        )

        if len(values) <= lookback:
            continue

        for i in range(
            lookback,
            len(values)
        ):

            X.append(
                values[
                    i-lookback:i
                ]
            )

            y.append(
                values[i]
            )

    X = np.array(X)
    y = np.array(y)

    X = X.reshape(
        X.shape[0],
        X.shape[1],
        1
    )

    return X, y

In [5]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

import numpy as np
import pandas as pd

In [6]:
def build_lstm_model(
    lookback
):

    model = Sequential([

        LSTM(
            64,
            input_shape=(
                lookback,
                1
            )
        ),

        Dense(1)

    ])

    model.compile(
        optimizer=Adam(
            learning_rate=0.001
        ),
        loss="mse"
    )

    return model

In [7]:
def train_lstm(
    train_df
):

    X_train, y_train = (
        create_sequences(
            train_df,
            LOOKBACK
        )
    )

    model = build_lstm_model(
        LOOKBACK
    )

    early_stop = EarlyStopping(

        monitor="loss",

        patience=3,

        restore_best_weights=True
    )

    model.fit(

        X_train,
        y_train,

        epochs=10,

        batch_size=512,

        verbose=0,

        callbacks=[
            early_stop
        ]
    )

    return model

In [8]:
def evaluate_lstm(
    train_df,
    test_df,
    horizon
):

    model = train_lstm(
        train_df
    )

    predictions = []

    for uid, group in train_df.groupby(
        "unique_id"
    ):

        history = (
            group["y"]
            .astype(float)
            .values
            .tolist()
        )

        if len(history) < LOOKBACK:
            continue

        uid_test = test_df[
            test_df["unique_id"] == uid
        ]

        for _ in range(
            min(horizon, len(uid_test))
        ):

            x = np.array(
                history[-LOOKBACK:]
            ).reshape(
                1,
                LOOKBACK,
                1
            )

            pred = float(
                model.predict(
                    x,
                    verbose=0
                )[0][0]
            )

            predictions.append(pred)

            history.append(pred)

    actuals = (
        test_df["y"]
        .astype(float)
        .values
    )

    min_len = min(
        len(actuals),
        len(predictions)
    )

    metrics = calculate_metrics(
        actuals[:min_len],
        predictions[:min_len]
    )

    return metrics

In [ ]:
for dataset_name in AVAILABLE_DATASETS:

    print("\n" + "=" * 80)
    print(dataset_name)

    run_key = (
        dataset_name,
        "LSTM"
    )

    if run_key in completed_runs:

        print("SKIP")
        continue

    try:

        (
            df,
            train_df,
            test_df,
            horizon
        ) = prepare_dataset(
            dataset_name
        )

        print("Training...")

        start = start_timer()

        metrics = evaluate_lstm(

            train_df,

            test_df,

            horizon
        )

        runtime = stop_timer(
            start
        )

        result = {

            "model":
            "LSTM",

            "dataset":
            dataset_name,

            "n_series":
            df["unique_id"].nunique(),

            "n_rows":
            len(df),

            "horizon":
            horizon,

            "runtime_seconds":
            runtime,

            **metrics
        }

        append_result(
            RESULT_FILE,
            result
        )

        print(
            f"MAE={metrics['MAE']:.4f}"
        )

    except Exception as e:

        print(
            f"ERROR: {e}"
        )


agri_medium
SKIP

agri_small
SKIP

energy_medium
SKIP

energy_small
SKIP

favorita-train_medium
SKIP

favorita-train_medium_cov
ERROR: No horizon configured for favorita-train_medium_cov

favorita-train_small
28
Training...


In [ ]:
results_df = pd.read_csv(
    RESULTS_FOLDER /
    RESULT_FILE
)

results_df.sort_values(
    [
        "dataset",
        "model"
    ]
)

,model,dataset,n_series,n_rows,horizon,runtime_seconds,MAE,RMSE,SMAPE


In [ ]:
results_df.groupby(
    "model"
)[
    [
        "MAE",
        "RMSE",
        "SMAPE",
        "runtime_seconds"
    ]
].mean().round(4)

,MAE,RMSE,SMAPE,runtime_seconds
model,,,,
